# Focused Event-Aligned Negative-Binomial GLMs

This notebook asks which interpretable variables explain single-neuron activity around three task moments: cue entry, R1 entry, and R2 entry. It uses event-aligned 40 ms spike counts, negative-binomial GLMs, trial-wise cross-validation, and full-versus-reduced model comparisons.

The analysis follows the same broad logic used in high-impact neural encoding work: model task variables and movement together, report held-out predictive performance, and interpret a block's contribution as unique predictive power after accounting for the other blocks (Musall et al., 2019; Steinmetz et al., 2019; Brunner et al., 2024; Harvey et al., 2012).


## Imports


In [ ]:
import os

# Keep BLAS/OpenMP single-threaded inside each worker. This must happen before
# importing numpy/scipy/statsmodels in a fresh kernel.
for _var in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[_var] = "1"

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

try:
    import seaborn as sns
except Exception:
    sns = None

project_root = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "analysisVR").is_dir() and (p / "baseVR").is_dir()
)
analysis_root = project_root / "analysisVR"
notebook_dir = analysis_root / "scripted_plotting" / "animal_6_analysis"
for _path in (project_root, analysis_root, notebook_dir):
    if str(_path) not in sys.path:
        sys.path.append(str(_path))

from baseVR.base_functionality import init_import_paths
init_import_paths()

from analytics_processing import analytics
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames, sessionlist_fullfnames_from_args
from CustomLogger import CustomLogger as Logger
import importlib
import choice_encoding_glm_parallel_utils as glm_utils

glm_utils = importlib.reload(glm_utils)
active_columns = glm_utils.active_columns
add_panel_fdr = glm_utils.add_panel_fdr
configure_single_thread_blas = glm_utils.configure_single_thread_blas
fit_nb = glm_utils.fit_nb
run_screen_parallel = glm_utils.run_screen_parallel

configure_single_thread_blas()
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
    warnings.filterwarnings("ignore", category=PerfectSeparationWarning)
except Exception:
    warnings.filterwarnings("ignore", message="Perfect separation or prediction detected.*")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 80)
Logger().init_logger(None, None, logging_level="WARNING")

mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
if sns is not None:
    sns.set_theme(context="paper", style="white", rc=mpl.rcParams)


## Configuration


In [ ]:
ANIMAL_IDS = [6]
PARADIGM_IDS = [1100]
SESSION_IDS = None
EXCL_SESSION_NAMES = [
    "2024-11-29_17-21_rYL006_P1100_LinearTrackStop_28min",
    "2025-01-22_17-51_rYL006_P1100_LinearTrackStop_5min",
    "2025-01-21_18-49_rYL006_P1100_LinearTrackStop_30min",
]

BIN_SECONDS = 0.04
BIN_US = 40_000
FDR_ALPHA = 0.05
RANDOM_STATE = 42

# First pass is intentionally capped. After the validation cell passes, set this
# to False and re-run the screen cells for the full analysis.
QUICK_TEST = False
MAX_INTERVAL_FITS = 350 if QUICK_TEST else None
MAX_TIME_FITS = 350 if QUICK_TEST else None
MAX_MOVEMENT_FITS = 350 if QUICK_TEST else None
MAX_MOVEMENT_VARIABLE_FITS = 350 if QUICK_TEST else None

MIN_TRIALS = 18
MIN_ROWS = 18
MIN_MEAN_COUNT = 0.02
MIN_NULL_DEVIANCE = 0.1
MAX_ABS_FDE = 10.0
N_CV_FOLDS = 5
PROGRESS_EVERY = 250 if QUICK_TEST else 2_000

CPU_COUNT = os.cpu_count() or 1
N_WORKERS = min(8 if QUICK_TEST else 64, max(1, CPU_COUNT - 2))
GROUPS_PER_TASK = 2 if QUICK_TEST else 4
PARALLEL_START_METHOD = "fork"

FIG_DIR = notebook_dir / "figures_choice_encoding_glm"
FIG_DIR.mkdir(parents=True, exist_ok=True)

INTERVAL_SPECS = {
    "cue_entry_interval": {"label": "Cue entry", "anchor": "Cue visible / cue-zone onset"},
    "R1_entry_interval": {"label": "R1 entry", "anchor": "R1 entry"},
    "R2_entry_interval": {"label": "R2 entry", "anchor": "R2 entry"},
}
INTERVAL_ORDER = list(INTERVAL_SPECS)
INTERVAL_LABELS = {k: v["label"] for k, v in INTERVAL_SPECS.items()}

MOVEMENT_COLUMNS = [
    "speed",
    "acceleration",
    "rotation",
    "rotation_acceleration",
    "forward_prop",
    "forward_rotation_corr",
    "head_angle",
    "head_angle_velocity",
    "movement_energy",
]

FEATURE_GROUPS = {
    "cue_identity": ["cue_binary"],
    "choice_stop": ["choice_stop"],
    "movement_body": MOVEMENT_COLUMNS,
}
FEATURE_ORDER = list(FEATURE_GROUPS)
FEATURE_COLORS = {
    "cue_identity": "#3A8F60",
    "choice_stop": "#B23A48",
    "movement_body": "#2878B5",
    "locomotion": "#D18F2F",
    "rotation": "#6D5BA6",
    "head_body_motion": "#4B7F7A",
}

MOVEMENT_DECOMP_GROUPS = {
    "locomotion": ["speed", "acceleration", "forward_prop"],
    "rotation": ["rotation", "rotation_acceleration", "forward_rotation_corr"],
    "head_body_motion": ["head_angle", "head_angle_velocity", "movement_energy"],
}

MOVEMENT_LABELS = {
    "speed": "Speed",
    "acceleration": "Acceleration",
    "forward_prop": "Forward proportion",
    "rotation": "Rotation",
    "rotation_acceleration": "Rotation acceleration",
    "forward_rotation_corr": "Forward/rotation coupling",
    "head_angle": "Head angle",
    "head_angle_velocity": "Head-angle velocity",
    "movement_energy": "Movement energy",
    "locomotion": "Locomotion",
    "rotation": "Rotation",
    "head_body_motion": "Head/body motion",
}

CONTINUOUS_COLS = set(MOVEMENT_COLUMNS) | {"trial_number_z", "rel_bin"}
GLM_SETTINGS = {
    "min_trials": MIN_TRIALS,
    "min_rows": MIN_ROWS,
    "min_mean_count": MIN_MEAN_COUNT,
    "min_null_deviance": MIN_NULL_DEVIANCE,
    "max_abs_fde": MAX_ABS_FDE,
    "n_cv_folds": N_CV_FOLDS,
    "random_state": RANDOM_STATE,
    "continuous_cols": CONTINUOUS_COLS,
}

print("Execution mode:", "QUICK TEST" if QUICK_TEST else "FULL ANALYSIS")
print(f"Workers: {N_WORKERS}; result figures: {FIG_DIR}")
print("Intervals:", [INTERVAL_LABELS[k] for k in INTERVAL_ORDER])


## Predictor Glossary And FDE

Primary model blocks:

- `cue_identity`: `cue_binary`, coded 0 for cue 1 and 1 for cue 2.
- `choice_stop`: reward-zone-specific stop/pass choice. At R1 entry this equals `choice_R1`; at R2 entry this equals `choice_R2`; at cue entry it is intentionally missing and therefore not fit.
- `movement_body`: continuous movement and body-motion variables: speed, acceleration, rotation, rotation acceleration, forward proportion, forward/rotation coupling, head angle, head-angle velocity, and movement energy.

Movement-only decomposition:

- `locomotion`: speed, acceleration, forward proportion.
- `rotation`: rotation, rotation acceleration, forward/rotation coupling.
- `head_body_motion`: head angle, head-angle velocity, movement energy.

Model metrics:

- `full_fde_cv`: held-out fraction deviance explained by the full model, `1 - full model deviance / null model deviance`.
- `standalone_fde_cv`: held-out FDE for baseline plus one block only.
- `delta_fde`: unique held-out contribution of a block, computed as full-model FDE minus FDE after removing that block.
- `p_value` and `q_value`: full-data likelihood-ratio test for the block, followed by Benjamini-Hochberg FDR correction.

Because behavioral variables can be correlated, `delta_fde` is a unique contribution given the other blocks, not an additive share of total variance.


## Load Analytics


In [ ]:
def _ensure_flat(df, name):
    if df is None:
        raise ValueError(f"Required analytic `{name}` is missing for all requested sessions.")
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out


def _unit_cols(df):
    return sorted([c for c in df.columns if str(c).startswith("Unit")])


def _normalize_unit(series):
    text = series.astype(str).str.extract(r"(?i)unit\s*0*(\d+)", expand=False)
    nums = pd.to_numeric(text, errors="coerce").combine_first(pd.to_numeric(series, errors="coerce"))
    return nums.map(lambda v: f"Unit{int(v):04d}" if pd.notna(v) else np.nan)


def _build_meta_map(spike_df):
    if spike_df is None:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])
    meta = _ensure_flat(spike_df, "SpikeClusterMetadata")
    area_col = next((c for c in ["fine_brain_area", "brain_area", "region", "area"] if c in meta.columns), None)
    unit_col = next((c for c in ["unit", "unit_id", "unit_name", "cluster_id_str", "cluster_id", "entry_id"] if c in meta.columns), None)
    if unit_col is None or "session_id" not in meta.columns:
        return pd.DataFrame(columns=["session_id", "unit", "brain_region"])
    meta["unit"] = _normalize_unit(meta[unit_col])
    meta["brain_region"] = meta[area_col].astype(str) if area_col else "Unknown"
    meta["brain_region"] = meta["brain_region"].replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"}).fillna("Unknown")
    return meta.dropna(subset=["session_id", "unit"]).groupby(["session_id", "unit"], as_index=False)["brain_region"].first()


session_dirs, _ = sessionlist_fullfnames_from_args(
    PARADIGM_IDS,
    ANIMAL_IDS,
    SESSION_IDS,
    excl_session_names=EXCL_SESSION_NAMES,
)
session_names = fullfnames2snames(session_dirs)

behavior_40 = _ensure_flat(
    analytics.get_analytics("Behavior40msAligned", session_names=session_names),
    "Behavior40msAligned",
)
fr40 = _ensure_flat(
    analytics.get_analytics("FiringRate40msHz", session_names=session_names),
    "FiringRate40msHz",
)
t0_events = _ensure_flat(
    analytics.get_analytics("TrialWiseT0Events40ms", session_names=session_names),
    "TrialWiseT0Events40ms",
)
spike_meta_raw = analytics.get_analytics("SpikeClusterMetadata", session_names=session_names)
meta_map = _build_meta_map(spike_meta_raw)
region_lookup = meta_map.set_index(["session_id", "unit"])["brain_region"].to_dict() if not meta_map.empty else {}
unit_cols = _unit_cols(fr40)

print(f"Sessions requested: {len(session_names):,}")
print(f"Sessions with 40 ms behavior: {behavior_40['session_id'].nunique():,}")
print(f"Sessions with 40 ms firing rates: {fr40['session_id'].nunique():,}")
print(f"Sessions with T0 intervals: {t0_events['session_id'].nunique():,}")
print(f"Unit columns: {len(unit_cols):,}")
print(f"Region labels available for {len(region_lookup):,} session-unit pairs")


## Predictor Engineering


In [ ]:
def _first_valid(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else np.nan


def _zscore_by_session(df, value_col):
    def zscore(s):
        std = s.std(ddof=0)
        return (s - s.mean()) / std if pd.notna(std) and std > 0 else pd.Series(0.0, index=s.index)
    return df.groupby("session_id", group_keys=False)[value_col].apply(zscore)


def _as_binary(series):
    values = pd.to_numeric(series, errors="coerce")
    return np.where(values.notna(), values.gt(0).astype(float), np.nan)


def _safe_counts(values):
    counts = np.rint(np.clip(pd.to_numeric(values, errors="coerce").fillna(0).to_numpy(dtype=float), 0, None))
    return counts.astype(np.int32)


BEHAVIOR_RENAMES = {
    "frame_raw_500msMedian": "speed",
    "frame_raw_abs_acc_500msMedian": "acceleration",
    "frame_YawPitch_abs_vel_sum_500msMedian": "rotation",
    "frame_YawPitch_abs_acc_sum_500msMedian": "rotation_acceleration",
    "frame_forward_prop": "forward_prop",
    "forward_vs_rotation_corr": "forward_rotation_corr",
    "head_angle_vel": "head_angle_velocity",
    "movement_energy_smooth5": "movement_energy",
}


def prepare_behavior_40(df):
    out = df.rename(columns=BEHAVIOR_RENAMES).copy()
    numeric_cols = [
        "from_ephys_timestamp",
        "to_ephys_timestamp",
        "trial_id",
        "cue",
        "trial_outcome",
        "track_zone_int",
        "frame_position",
        *MOVEMENT_COLUMNS,
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "upcoming_choice" in out.columns:
        upcoming = pd.to_numeric(out["upcoming_choice"], errors="coerce")
        out["upcoming_choice_skip"] = upcoming.lt(0).astype(float)
        out["upcoming_choice_stop"] = upcoming.gt(0).astype(float)
    if "reward_window" in out.columns:
        reward_window = pd.to_numeric(out["reward_window"], errors="coerce")
        out["reward_window_pre"] = reward_window.lt(0).astype(float)
        out["reward_window_post"] = reward_window.gt(0).astype(float)
    if "cue_visible" in out.columns:
        cue_visible = pd.to_numeric(out["cue_visible"], errors="coerce")
        out["cue_visible_1"] = cue_visible.eq(1).astype(float)
        out["cue_visible_2"] = cue_visible.eq(2).astype(float)
    for raw_col, out_col in [
        ("lick_detected", "lick"),
        ("reward-sound_detected", "reward_sound"),
        ("reward-valve-open_detected", "reward_valve"),
    ]:
        if raw_col in out.columns:
            out[out_col] = pd.to_numeric(out[raw_col], errors="coerce").fillna(0).clip(0, 1)
    return out


def build_trial_table(t0_df, behavior_df):
    t0_trial = (
        t0_df[["session_id", "trial_id", "cue", "trial_outcome", "choice_R1", "choice_R2"]]
        .dropna(subset=["session_id", "trial_id"])
        .copy()
    )
    t0_trial["trial_id"] = pd.to_numeric(t0_trial["trial_id"], errors="coerce")
    for col in ["cue", "trial_outcome"]:
        t0_trial[col] = pd.to_numeric(t0_trial[col], errors="coerce")
    t0_trial["choice_R1"] = _as_binary(t0_trial["choice_R1"])
    t0_trial["choice_R2"] = _as_binary(t0_trial["choice_R2"])
    t0_trial = t0_trial.groupby(["session_id", "trial_id"], as_index=False).agg(_first_valid)

    beh_trial = behavior_df[["session_id", "trial_id", "choice_R1", "choice_R2"]].dropna(subset=["session_id", "trial_id"]).copy()
    beh_trial["trial_id"] = pd.to_numeric(beh_trial["trial_id"], errors="coerce")
    beh_trial["choice_R1"] = _as_binary(beh_trial["choice_R1"])
    beh_trial["choice_R2"] = _as_binary(beh_trial["choice_R2"])
    beh_trial = beh_trial.groupby(["session_id", "trial_id"], as_index=False).agg(_first_valid)

    cmp = t0_trial.merge(beh_trial, on=["session_id", "trial_id"], how="inner", suffixes=("_t0", "_behavior"))
    mismatch = pd.Series(False, index=cmp.index)
    for col in ["choice_R1", "choice_R2"]:
        mismatch |= cmp[f"{col}_t0"].notna() & cmp[f"{col}_behavior"].notna() & ~np.isclose(cmp[f"{col}_t0"], cmp[f"{col}_behavior"])
    if mismatch.any():
        warnings.warn(
            f"{int(mismatch.sum())} trial-level choice mismatches between TrialWiseT0Events40ms and Behavior40msAligned; "
            "using TrialWiseT0Events40ms choices."
        )

    out = t0_trial.sort_values(["session_id", "trial_id"]).copy()
    out["cue_binary"] = np.where(out["cue"].eq(1), 0.0, np.where(out["cue"].eq(2), 1.0, np.nan))
    out["outcome_binary"] = np.where(out["trial_outcome"].notna(), out["trial_outcome"].gt(0).astype(float), np.nan)
    out["trial_number"] = out.groupby("session_id").cumcount() + 1
    out["trial_number_z"] = _zscore_by_session(out, "trial_number")
    return out, cmp.loc[mismatch]


behavior_40_features = prepare_behavior_40(behavior_40)
trial_table, choice_mismatch_table = build_trial_table(t0_events, behavior_40)

display(trial_table.head())
if not choice_mismatch_table.empty:
    display(choice_mismatch_table.head())


## Build Event-Aligned Model Table


In [ ]:
def _interval_bins(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, np.ndarray):
        return [int(v) for v in value.tolist()]
    if isinstance(value, (list, tuple)):
        return [int(v) for v in value]
    return []


def _available_feature_groups(df, groups):
    return {
        name: [c for c in cols if c in df.columns]
        for name, cols in groups.items()
        if any(c in df.columns for c in cols)
    }


def expand_event_intervals(t0_df):
    rows = []
    keep = t0_df.dropna(subset=["session_id", "trial_id", "t0_event_name", "t0"]).copy()
    for row in keep.itertuples(index=False):
        session_id = getattr(row, "session_id")
        trial_id = float(getattr(row, "trial_id"))
        t0 = float(getattr(row, "t0"))
        t0_event_name = getattr(row, "t0_event_name")
        for interval_name, spec in INTERVAL_SPECS.items():
            interval = getattr(row, interval_name, None)
            bins = _interval_bins(getattr(row, f"{interval_name}_bins", None))
            if pd.isna(interval) or not bins:
                continue
            for rel_bin in bins:
                rows.append({
                    "session_id": session_id,
                    "trial_id": trial_id,
                    "t0_event_name": t0_event_name,
                    "interval_name": interval_name,
                    "interval_label": spec["label"],
                    "interval_anchor": spec["anchor"],
                    "event_t0_us": t0,
                    "rel_bin": int(rel_bin),
                    "rel_time_s": float(rel_bin * BIN_SECONDS),
                    "from_ephys_timestamp": float(t0 + rel_bin * BIN_US),
                    "to_ephys_timestamp": float(t0 + (rel_bin + 1) * BIN_US),
                    "exposure": BIN_SECONDS,
                })
    return pd.DataFrame.from_records(rows)


fr_counts = fr40[["session_id", "from_ephys_timestamp", "to_ephys_timestamp", *unit_cols]].copy()
fr_counts["from_ephys_timestamp"] = pd.to_numeric(fr_counts["from_ephys_timestamp"], errors="coerce")
fr_counts["to_ephys_timestamp"] = pd.to_numeric(fr_counts["to_ephys_timestamp"], errors="coerce")
for unit in unit_cols:
    fr_counts[unit] = _safe_counts(pd.to_numeric(fr_counts[unit], errors="coerce") / 25.0)

behavior_predictor_drop = {
    "paradigm_id",
    "animal_id",
    "entry_id",
    "trial_id",
    "cue",
    "trial_outcome",
    "choice_R1",
    "choice_R2",
    "frame_ephys_timestamp",
    "frame_pc_timestamp",
    "to_ephys_timestamp",
}
behavior_predictors = behavior_40_features.drop(columns=[c for c in behavior_predictor_drop if c in behavior_40_features.columns])
behavior_predictors["from_ephys_timestamp"] = pd.to_numeric(behavior_predictors["from_ephys_timestamp"], errors="coerce")

event_index = expand_event_intervals(t0_events)
event_bin_table = (
    event_index
    .merge(fr_counts.drop(columns="to_ephys_timestamp"), on=["session_id", "from_ephys_timestamp"], how="inner")
    .merge(behavior_predictors, on=["session_id", "from_ephys_timestamp"], how="left")
    .merge(trial_table, on=["session_id", "trial_id"], how="left")
)

interval = event_bin_table["interval_name"].astype(str)
event_bin_table["choice_stop"] = np.select(
    [interval.eq("R1_entry_interval"), interval.eq("R2_entry_interval")],
    [event_bin_table["choice_R1"], event_bin_table["choice_R2"]],
    default=np.nan,
).astype(float)
event_bin_table["choice_stop_label"] = np.select(
    [event_bin_table["choice_stop"].eq(1), event_bin_table["choice_stop"].eq(0)],
    ["Stop", "Pass"],
    default="Not modeled",
)
event_bin_table["baseline_constant"] = 1.0

BASELINE_INTERVAL = ["rel_bin", "trial_number_z"]
BASELINE_TIME = ["trial_number_z"]

valid = event_bin_table["exposure"].gt(0) & event_bin_table["trial_id"].notna()
event_bin_table = event_bin_table.loc[valid].reset_index(drop=True)
event_bin_table["trial_instance"] = event_bin_table["session_id"].astype(str) + "::" + event_bin_table["trial_id"].astype(str)

if QUICK_TEST:
    group_priority = (
        event_bin_table.groupby(["session_id", "interval_name"], as_index=False)
        .agg(priority_trials=("trial_id", "nunique"), priority_rows=("trial_id", "size"))
    )
    group_priority["priority_eligible"] = group_priority["priority_trials"].ge(MIN_TRIALS)
    event_bin_table = (
        event_bin_table.merge(group_priority, on=["session_id", "interval_name"], how="left")
        .sort_values(
            ["priority_eligible", "priority_trials", "priority_rows", "session_id", "interval_name", "rel_bin"],
            ascending=[False, False, False, True, True, True],
        )
        .drop(columns=["priority_eligible", "priority_trials", "priority_rows"])
        .reset_index(drop=True)
    )

feature_groups = _available_feature_groups(event_bin_table, FEATURE_GROUPS)
movement_decomp_groups = _available_feature_groups(event_bin_table, MOVEMENT_DECOMP_GROUPS)
movement_variable_groups = {
    col: [col]
    for col in MOVEMENT_COLUMNS
    if col in event_bin_table.columns
}

coverage = (
    event_bin_table.groupby(["interval_name", "interval_label"], as_index=False)
    .agg(
        n_rows=("trial_id", "size"),
        n_trials=("trial_instance", "nunique"),
        n_sessions=("session_id", "nunique"),
        stop_trials=("choice_stop", lambda s: int(s.eq(1).sum())),
        pass_trials=("choice_stop", lambda s: int(s.eq(0).sum())),
        min_rel_s=("rel_time_s", "min"),
        max_rel_s=("rel_time_s", "max"),
    )
)
coverage["interval_name"] = pd.Categorical(coverage["interval_name"], INTERVAL_ORDER, ordered=True)
coverage = coverage.sort_values("interval_name")

print(f"Event-index rows before merges: {len(event_index):,}")
print(f"Event-bin model table: {event_bin_table.shape[0]:,} rows x {event_bin_table.shape[1]:,} columns")
print("Primary feature groups:", {k: len(v) for k, v in feature_groups.items()})
print("Movement decomposition groups:", {k: len(v) for k, v in movement_decomp_groups.items()})
print("Movement variable groups:", {k: len(v) for k, v in movement_variable_groups.items()})
display(coverage)
display(event_bin_table[["session_id", "trial_id", "interval_label", "rel_bin", "exposure", "cue_binary", "choice_stop", "choice_R1", "choice_R2", *unit_cols[:3]]].head())


## Run Event-Aligned NB-GLM Screens


In [ ]:
def _run_screen(base_df, group_keys, panel, baseline_cols, max_fits, groups):
    return run_screen_parallel(
        base_df=base_df,
        unit_columns=unit_cols,
        group_keys=group_keys,
        feature_groups=groups,
        panel=panel,
        settings=GLM_SETTINGS,
        region_lookup=region_lookup,
        max_fits=max_fits,
        baseline_cols=baseline_cols,
        n_workers=N_WORKERS,
        groups_per_task=GROUPS_PER_TASK,
        start_method=PARALLEL_START_METHOD,
        progress_every=PROGRESS_EVERY,
    )


print("Running interval-pooled primary models...")
interval_results_raw = _run_screen(
    event_bin_table,
    ["session_id", "interval_name", "interval_label"],
    panel="interval",
    baseline_cols=BASELINE_INTERVAL,
    max_fits=MAX_INTERVAL_FITS,
    groups=feature_groups,
)

print("Running relative-time-resolved primary models...")
time_results_raw = _run_screen(
    event_bin_table,
    ["session_id", "interval_name", "interval_label", "rel_bin", "rel_time_s"],
    panel="time_resolved",
    baseline_cols=BASELINE_TIME,
    max_fits=MAX_TIME_FITS,
    groups=feature_groups,
)

print("Running movement decomposition models...")
movement_decomp_results_raw = _run_screen(
    event_bin_table,
    ["session_id", "interval_name", "interval_label"],
    panel="movement_decomp",
    baseline_cols=BASELINE_INTERVAL,
    max_fits=MAX_MOVEMENT_FITS,
    groups=movement_decomp_groups,
)

print("Running movement variable ranking models...")
movement_variable_results_raw = _run_screen(
    event_bin_table,
    ["session_id", "interval_name", "interval_label"],
    panel="movement_variables",
    baseline_cols=BASELINE_INTERVAL,
    max_fits=MAX_MOVEMENT_VARIABLE_FITS,
    groups=movement_variable_groups,
)

def _ensure_result_schema(df):
    out = df.copy()
    if "standalone_fde_cv" not in out.columns:
        out["standalone_fde_cv"] = np.nan
    return out


interval_results = _ensure_result_schema(add_panel_fdr(interval_results_raw, ["panel", "interval_name"], FDR_ALPHA))
time_results = _ensure_result_schema(add_panel_fdr(time_results_raw, ["panel", "interval_name"], FDR_ALPHA))
movement_decomp_results = _ensure_result_schema(add_panel_fdr(movement_decomp_results_raw, ["panel", "interval_name"], FDR_ALPHA))
movement_variable_results = _ensure_result_schema(add_panel_fdr(movement_variable_results_raw, ["panel", "interval_name"], FDR_ALPHA))
all_feature_results = pd.concat(
    [interval_results, time_results, movement_decomp_results, movement_variable_results],
    ignore_index=True,
)

display(all_feature_results.head(12))
print("Result rows:", {name: len(df) for name, df in [
    ("interval", interval_results),
    ("time_resolved", time_results),
    ("movement_decomp", movement_decomp_results),
    ("movement_variables", movement_variable_results),
]})


## Minimal Diagnostics


In [ ]:
def save_figure(fig, stem):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved {png.relative_to(notebook_dir)} and {pdf.relative_to(notebook_dir)}")


diag_df = all_feature_results.copy()
fig, axes = plt.subplots(1, 3, figsize=(8.2, 2.4), constrained_layout=True)
if diag_df.empty:
    axes[0].text(0.5, 0.5, "No eligible fits", ha="center", va="center")
else:
    for panel, sub in diag_df.groupby("panel"):
        axes[0].scatter(sub["mean_count"], sub["var_count"], s=5, alpha=0.35, label=panel)
    max_mv = float(np.nanmax(diag_df[["mean_count", "var_count"]].to_numpy()))
    max_mv = max(max_mv, 1e-2)
    axes[0].plot([1e-3, max_mv], [1e-3, max_mv], color="black", lw=0.8, ls="--")
    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("Mean count")
    axes[0].set_ylabel("Variance")
    axes[0].legend(frameon=False, loc="best")
    axes[1].hist(diag_df["alpha_nb"].dropna(), bins=40, color="#2878B5", alpha=0.85)
    axes[1].set_xlabel("NB dispersion alpha")
    axes[1].set_ylabel("Fits")
    counts = diag_df.groupby("panel").size().reindex(["interval", "time_resolved", "movement_decomp", "movement_variables"]).dropna()
    axes[2].bar(counts.index, counts.values, color="#595959")
    axes[2].tick_params(axis="x", rotation=35)
    axes[2].set_ylabel("Result rows")
for ax, title in zip(axes, ["Mean-variance", "Dispersion", "Eligible fits"]):
    ax.set_title(title)
save_figure(fig, "fig00_nb_glm_diagnostics")
plt.show()


## Figure 1: Event Windows And Coverage


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), gridspec_kw={"width_ratios": [1.2, 1.0]}, constrained_layout=True)
plot_cov = coverage.copy()
plot_cov["interval_label"] = pd.Categorical(plot_cov["interval_label"], [INTERVAL_LABELS[k] for k in INTERVAL_ORDER], ordered=True)
plot_cov = plot_cov.sort_values("interval_label")
y = np.arange(len(plot_cov))
axes[0].hlines(y, plot_cov["min_rel_s"], plot_cov["max_rel_s"] + BIN_SECONDS, color="#404040", lw=5, alpha=0.8)
axes[0].axvline(0, color="#B23A48", lw=1.0)
axes[0].set_yticks(y)
axes[0].set_yticklabels(plot_cov["interval_label"])
axes[0].invert_yaxis()
axes[0].set_xlabel("Time from event (s)")
axes[0].set_title("Aligned windows")

axes[1].barh(y, plot_cov["n_trials"], color="#2878B5", alpha=0.85)
axes[1].set_yticks(y)
axes[1].set_yticklabels([])
axes[1].invert_yaxis()
axes[1].set_xlabel("Trials")
axes[1].set_title("Coverage")
for yi, n_sess in zip(y, plot_cov["n_sessions"]):
    axes[1].text(0, yi, f" {int(n_sess)} sess", va="center", ha="left", color="white", fontsize=6)

save_figure(fig, "fig01_event_windows_coverage")
plt.show()


## Figure 2: NB-GLM Design Schematic


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2), constrained_layout=True)
ax.axis("off")
blocks = [
    (0.02, 0.56, 0.22, 0.28, "TrialWiseT0Events40ms\n3 event intervals + choices"),
    (0.02, 0.16, 0.22, 0.28, "FiringRate40msHz\n40 ms spike counts"),
    (0.30, 0.36, 0.22, 0.28, "Event-aligned\nunit x bin table"),
    (0.58, 0.36, 0.17, 0.28, "Full NB-GLM\ncue + stop/pass + movement"),
    (0.80, 0.36, 0.17, 0.28, "Reduced NB-GLM\none block held out"),
]
for x, y0, w, h, text in blocks:
    rect = mpl.patches.FancyBboxPatch((x, y0), w, h, boxstyle="round,pad=0.015", linewidth=0.8, edgecolor="#333333", facecolor="#F7F7F7")
    ax.add_patch(rect)
    ax.text(x + w / 2, y0 + h / 2, text, ha="center", va="center", fontsize=8)
for x0, y0, x1, y1 in [(0.24, 0.70, 0.30, 0.50), (0.24, 0.30, 0.30, 0.50), (0.52, 0.50, 0.58, 0.50), (0.75, 0.50, 0.80, 0.50)]:
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="#333333", lw=0.9))

feature_text = "Primary blocks:\n" + "\n".join([
    "cue_identity: cue 1 vs cue 2",
    "choice_stop: R1 stop/pass or R2 stop/pass",
    "movement_body: kinematics + head/body motion",
])
ax.text(0.30, 0.15, feature_text, ha="left", va="center", fontsize=7, color="#333333", linespacing=1.4)
ax.text(0.58, 0.18, "Effect size = held-out delta FDE; standalone FDE is also reported; p = LRT; q = BH-FDR", ha="left", va="center", fontsize=7, color="#333333")
save_figure(fig, "fig02_glm_design_schematic")
plt.show()


## Figure 3: Interval-Level Feature Encoding


In [ ]:
def _ordered_feature_index(values, order):
    present = set(values)
    return [f for f in order if f in present]


interval_summary = (
    interval_results.groupby(["interval_name", "interval_label", "feature_group"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        median_standalone_fde=("standalone_fde_cv", "median"),
        median_full_fde=("full_fde_cv", "median"),
        n_fits=("unit", "size"),
        n_significant=("is_significant", "sum"),
    )
)
interval_summary["interval_name"] = pd.Categorical(interval_summary["interval_name"], INTERVAL_ORDER, ordered=True)
interval_summary = interval_summary.sort_values("interval_name")
interval_names_present = [k for k in INTERVAL_ORDER if k in set(interval_summary["interval_name"].astype(str))]
feature_y = _ordered_feature_index(interval_summary["feature_group"], FEATURE_ORDER)

piv_sig = (
    interval_summary.pivot_table(index="feature_group", columns="interval_name", values="frac_significant", observed=False)
    .reindex(index=feature_y, columns=interval_names_present)
)
piv_delta = (
    interval_summary.pivot_table(index="feature_group", columns="interval_name", values="median_delta_fde", observed=False)
    .reindex(index=feature_y, columns=interval_names_present)
)
piv_standalone = (
    interval_summary.pivot_table(index="feature_group", columns="interval_name", values="median_standalone_fde", observed=False)
    .reindex(index=feature_y, columns=interval_names_present)
)

fig_height = max(2.6, 1.0 + 0.34 * max(1, len(feature_y)))
fig, axes = plt.subplots(1, 3, figsize=(9.2, fig_height), constrained_layout=True)
if interval_summary.empty or not feature_y or not interval_names_present:
    for ax in axes:
        ax.text(0.5, 0.5, "No interval GLM results", ha="center", va="center")
        ax.axis("off")
else:
    for ax, data, title, cbar_label, cmap in [
        (axes[0], piv_delta, "Unique held-out contribution", "delta FDE", "viridis"),
        (axes[1], piv_standalone, "Standalone prediction", "FDE", "crest" if sns is not None else "Blues"),
        (axes[2], piv_sig, "Fraction significant", "Fraction", "magma"),
    ]:
        if sns is not None:
            sns.heatmap(data, ax=ax, cmap=cmap, linewidths=0.25, linecolor="white", cbar_kws={"label": cbar_label})
        else:
            im = ax.imshow(data.to_numpy(dtype=float), aspect="auto", cmap=cmap)
            fig.colorbar(im, ax=ax, label=cbar_label)
        ax.set_title(title)
        ax.set_xlabel("Interval")
        ax.set_ylabel("Feature block")
        ax.set_xticklabels([INTERVAL_LABELS[k] for k in interval_names_present], rotation=45, ha="right")

save_figure(fig, "fig03_interval_feature_encoding")
plt.show()
if not interval_summary.empty:
    display(interval_summary.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).head(20))


## Figure 4: Time-Resolved Stop/Pass Encoding


In [ ]:
time_summary = (
    time_results.groupby(["interval_name", "interval_label", "rel_bin", "rel_time_s", "feature_group"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        median_standalone_fde=("standalone_fde_cv", "median"),
        n_fits=("unit", "size"),
    )
)
choice_time = time_summary[time_summary["feature_group"].eq("choice_stop")].copy()
choice_time["interval_name"] = pd.Categorical(choice_time["interval_name"], INTERVAL_ORDER, ordered=True)
choice_time = choice_time.sort_values(["interval_name", "rel_bin"])

interval_names_present = [k for k in ["R1_entry_interval", "R2_entry_interval"] if k in set(choice_time["interval_name"].astype(str))]
fig, axes = plt.subplots(max(len(interval_names_present), 1), 1, figsize=(7.2, max(2.2, 1.2 * len(interval_names_present))), sharex=False, constrained_layout=True)
axes = np.atleast_1d(axes)
if choice_time.empty:
    axes[0].text(0.5, 0.5, "No time-resolved stop/pass fits", ha="center", va="center")
    axes[0].axis("off")
else:
    vmax = max(0.05, float(choice_time["frac_significant"].max()))
    for ax, interval_name in zip(axes, interval_names_present):
        sub = choice_time[choice_time["interval_name"].astype(str).eq(interval_name)]
        x = sub["rel_time_s"].to_numpy(dtype=float)
        y = sub["frac_significant"].to_numpy(dtype=float)
        delta = sub["median_delta_fde"].to_numpy(dtype=float)
        ax.axvline(0, color="#B23A48", lw=0.8)
        ax.plot(x, y, color="#B23A48", lw=1.4, label="Fraction significant")
        ax.fill_between(x, 0, y, color="#B23A48", alpha=0.18)
        ax2 = ax.twinx()
        ax2.plot(x, delta, color="#2878B5", lw=1.1, label="Median delta FDE")
        ax2.tick_params(axis="y", labelsize=6, colors="#2878B5")
        ax2.spines["right"].set_visible(True)
        ax.set_ylim(0, vmax * 1.08)
        ax.set_ylabel(INTERVAL_LABELS[interval_name])
        ax.tick_params(axis="y", labelsize=6, colors="#B23A48")
    axes[-1].set_xlabel("Time from event (s)")
    axes[0].set_title("Time-resolved reward-zone stop/pass encoding")

save_figure(fig, "fig04_time_resolved_stop_pass_encoding")
plt.show()
if not choice_time.empty:
    display(choice_time.sort_values(["frac_significant", "median_delta_fde"], ascending=[False, False]).head(20))


## Movement-Only Decomposition


In [ ]:
movement_decomp_summary = (
    movement_decomp_results.groupby(["interval_name", "interval_label", "feature_group"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        median_standalone_fde=("standalone_fde_cv", "median"),
        n_fits=("unit", "size"),
        n_significant=("is_significant", "sum"),
    )
)
movement_decomp_summary["interval_name"] = pd.Categorical(movement_decomp_summary["interval_name"], INTERVAL_ORDER, ordered=True)
movement_decomp_summary = movement_decomp_summary.sort_values("interval_name")

movement_variable_summary = (
    movement_variable_results.groupby(["interval_name", "interval_label", "feature_group"], as_index=False)
    .agg(
        frac_significant=("is_significant", "mean"),
        median_delta_fde=("delta_fde", "median"),
        median_standalone_fde=("standalone_fde_cv", "median"),
        n_fits=("unit", "size"),
        n_significant=("is_significant", "sum"),
    )
)
movement_variable_summary["interval_name"] = pd.Categorical(movement_variable_summary["interval_name"], INTERVAL_ORDER, ordered=True)
movement_variable_summary = movement_variable_summary.sort_values("interval_name")
movement_variable_summary["movement_label"] = movement_variable_summary["feature_group"].map(MOVEMENT_LABELS).fillna(movement_variable_summary["feature_group"])

decomp_y = [k for k in MOVEMENT_DECOMP_GROUPS if k in set(movement_decomp_summary["feature_group"])]
var_order = [c for c in MOVEMENT_COLUMNS if c in set(movement_variable_summary["feature_group"])]
interval_names_present = [k for k in INTERVAL_ORDER if k in set(movement_decomp_summary["interval_name"].astype(str))]

decomp_delta = (
    movement_decomp_summary.pivot_table(index="feature_group", columns="interval_name", values="median_delta_fde", observed=False)
    .reindex(index=decomp_y, columns=interval_names_present)
)
var_delta = (
    movement_variable_summary.pivot_table(index="feature_group", columns="interval_name", values="median_delta_fde", observed=False)
    .reindex(index=var_order, columns=interval_names_present)
)
var_rank = (
    movement_variable_summary.groupby("feature_group", as_index=False)
    .agg(overall_delta_fde=("median_delta_fde", "median"), overall_standalone_fde=("median_standalone_fde", "median"))
    .sort_values("overall_delta_fde", ascending=False)
)
var_rank["movement_label"] = var_rank["feature_group"].map(MOVEMENT_LABELS).fillna(var_rank["feature_group"])

fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4), constrained_layout=True)
if movement_decomp_summary.empty:
    for ax in axes:
        ax.text(0.5, 0.5, "No movement-only GLM results", ha="center", va="center")
        ax.axis("off")
else:
    if sns is not None:
        sns.heatmap(decomp_delta, ax=axes[0], cmap="viridis", linewidths=0.25, linecolor="white", cbar_kws={"label": "delta FDE"})
        sns.heatmap(var_delta, ax=axes[1], cmap="viridis", linewidths=0.25, linecolor="white", cbar_kws={"label": "delta FDE"})
    else:
        im0 = axes[0].imshow(decomp_delta.to_numpy(dtype=float), aspect="auto", cmap="viridis")
        fig.colorbar(im0, ax=axes[0], label="delta FDE")
        im1 = axes[1].imshow(var_delta.to_numpy(dtype=float), aspect="auto", cmap="viridis")
        fig.colorbar(im1, ax=axes[1], label="delta FDE")
    axes[0].set_title("Movement sub-blocks")
    axes[0].set_xlabel("Interval")
    axes[0].set_ylabel("Movement type")
    axes[0].set_yticklabels([MOVEMENT_LABELS.get(v.get_text(), v.get_text()) for v in axes[0].get_yticklabels()], rotation=0)
    axes[0].set_xticklabels([INTERVAL_LABELS[k] for k in interval_names_present], rotation=45, ha="right")
    axes[1].set_title("Single movement variables")
    axes[1].set_xlabel("Interval")
    axes[1].set_ylabel("Variable")
    axes[1].set_yticklabels([MOVEMENT_LABELS.get(v.get_text(), v.get_text()) for v in axes[1].get_yticklabels()], rotation=0)
    axes[1].set_xticklabels([INTERVAL_LABELS[k] for k in interval_names_present], rotation=45, ha="right")

    top_rank = var_rank.head(8).sort_values("overall_delta_fde")
    axes[2].barh(np.arange(len(top_rank)), top_rank["overall_delta_fde"], color="#2878B5", alpha=0.9)
    axes[2].set_yticks(np.arange(len(top_rank)))
    axes[2].set_yticklabels(top_rank["movement_label"])
    axes[2].set_xlabel("Median delta FDE")
    axes[2].set_title("Overall movement ranking")

save_figure(fig, "fig05_movement_decomposition")
plt.show()

movement_lines = []
if not movement_decomp_summary.empty:
    top_decomp = movement_decomp_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).iloc[0]
    movement_lines.append(
        f"- Most influential movement type: **{MOVEMENT_LABELS.get(top_decomp.feature_group, top_decomp.feature_group)}** "
        f"in **{top_decomp.interval_label}** (median delta FDE {top_decomp.median_delta_fde:.4f})."
    )
if not var_rank.empty:
    top_var = var_rank.iloc[0]
    movement_lines.append(
        f"- Most influential single movement variable overall: **{top_var.movement_label}** "
        f"(median delta FDE {top_var.overall_delta_fde:.4f})."
    )
if not movement_lines:
    movement_lines.append("- No eligible movement-only GLM results were produced.")
display(Markdown("\n".join(movement_lines)))
if not movement_decomp_summary.empty:
    display(movement_decomp_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).head(12))
if not movement_variable_summary.empty:
    display(movement_variable_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).head(20))


## Figure 6: Single-Neuron Exemplar


In [ ]:
def unit_model_frame(base_df, unit, predictors):
    keep_cols = list(dict.fromkeys([
        "session_id",
        "trial_id",
        "exposure",
        "trial_number",
        "trial_number_z",
        "interval_name",
        "interval_label",
        "rel_bin",
        "rel_time_s",
        "cue",
        "cue_binary",
        "choice_stop",
        "choice_stop_label",
        "choice_R1",
        "choice_R2",
        *predictors,
    ]))
    keep_cols = [c for c in keep_cols if c in base_df.columns]
    out = base_df[keep_cols].copy()
    out["unit"] = unit
    out["brain_region"] = [region_lookup.get((s, unit), "Unknown") for s in out["session_id"]]
    out["spike_count"] = pd.to_numeric(base_df[unit], errors="coerce")
    return out


def _source_for_result(row):
    source = event_bin_table[
        event_bin_table["session_id"].eq(row.session_id)
        & event_bin_table["interval_name"].eq(row.interval_name)
    ]
    predictors = sorted({c for cols in feature_groups.values() for c in cols} | set(BASELINE_INTERVAL))
    return unit_model_frame(source, row.unit, predictors), BASELINE_INTERVAL


def prediction_df(row):
    model_df, baseline = _source_for_result(row)
    groups = {name: active_columns(model_df, cols) for name, cols in feature_groups.items()}
    groups = {name: cols for name, cols in groups.items() if cols}
    full_cols = active_columns(model_df, baseline) + sorted({c for cols in groups.values() for c in cols})
    target_cols = groups.get(row.feature_group, [])
    reduced_cols = [c for c in full_cols if c not in target_cols]
    full_res, full_x, _, _ = fit_nb(model_df, full_cols, model_df, GLM_SETTINGS)
    reduced_res, reduced_x, _, _ = fit_nb(model_df, reduced_cols, model_df, GLM_SETTINGS)
    offset = np.log(np.clip(model_df["exposure"].to_numpy(dtype=float), 1e-9, None))
    out = model_df.copy()
    out["pred_full"] = np.clip(full_res.predict(full_x, offset=offset), 1e-9, None)
    out["pred_reduced"] = np.clip(reduced_res.predict(reduced_x, offset=offset), 1e-9, None)
    out["plot_label"] = np.where(
        out["choice_stop"].notna(),
        out["choice_stop_label"],
        np.where(out["cue_binary"].eq(0), "Cue 1", np.where(out["cue_binary"].eq(1), "Cue 2", "Other")),
    )
    return out


candidate = interval_results[interval_results["is_significant"] & interval_results["feature_group"].eq("choice_stop")]
if candidate.empty:
    candidate = interval_results[interval_results["is_significant"]]
if candidate.empty:
    candidate = interval_results.dropna(subset=["delta_fde"]).sort_values("delta_fde", ascending=False).head(1)
else:
    candidate = candidate.sort_values(["delta_fde", "full_fde_cv"], ascending=False).head(1)

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
if candidate.empty:
    axes[0].text(0.5, 0.5, "No eligible exemplar", ha="center", va="center")
    axes[1].axis("off")
else:
    ex = candidate.iloc[0]
    pred = prediction_df(ex)
    pred["observed_rate"] = pred["spike_count"] / pred["exposure"]
    pred["pred_full_rate"] = pred["pred_full"] / pred["exposure"]
    pred["pred_reduced_rate"] = pred["pred_reduced"] / pred["exposure"]
    plot = (
        pred.groupby(["rel_time_s", "plot_label"], as_index=False)
        .agg(
            observed=("observed_rate", "mean"),
            full=("pred_full_rate", "mean"),
            reduced=("pred_reduced_rate", "mean"),
            n=("trial_id", "nunique"),
        )
        .sort_values("rel_time_s")
    )
    for label, sub in plot.groupby("plot_label"):
        axes[0].plot(sub["rel_time_s"], sub["observed"], marker="o", ms=2.2, lw=0.9, label=label)
        axes[1].plot(sub["rel_time_s"], sub["full"], lw=1.2, label=f"{label} full")
        axes[1].plot(sub["rel_time_s"], sub["reduced"], lw=1.0, ls="--", alpha=0.8, label=f"{label} reduced")
    for ax in axes:
        ax.axvline(0, color="#B23A48", lw=0.8)
        ax.set_xlabel("Time from event (s)")
        ax.set_ylabel("Spike rate (Hz)")
    axes[0].set_title("Observed condition rate")
    axes[1].set_title("Full vs reduced predictions")
    axes[1].legend(frameon=False, ncol=1, loc="best", fontsize=6)
    axes[0].legend(frameon=False, loc="best", fontsize=6)
    title = f"{ex.unit} | {ex.session_id} | {ex.interval_label} | {ex.feature_group} delta FDE={ex.delta_fde:.3f}, q={ex.q_value:.3g}"
    fig.suptitle(title, fontsize=8)

save_figure(fig, "fig06_single_neuron_exemplar")
plt.show()
if not candidate.empty:
    display(candidate)


## Figure 7: Session And Region Summaries


In [ ]:
summary_source = interval_results.copy()
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.0), constrained_layout=True)
if summary_source.empty:
    axes[0].text(0.5, 0.5, "No interval results", ha="center", va="center")
    axes[1].axis("off")
else:
    session_feature_summary = (
        summary_source.groupby(["session_id", "feature_group"], as_index=False)
        .agg(
            median_delta_fde=("delta_fde", "median"),
            median_full_fde=("full_fde_cv", "median"),
            frac_significant=("is_significant", "mean"),
            n_units=("unit", "nunique"),
        )
        .sort_values("session_id")
    )
    x_labels = sorted(session_feature_summary["session_id"].unique())
    x_lookup = {s: i for i, s in enumerate(x_labels)}
    for feature in FEATURE_ORDER:
        sub = session_feature_summary[session_feature_summary["feature_group"].eq(feature)]
        if sub.empty:
            continue
        axes[0].plot(
            [x_lookup[s] for s in sub["session_id"]],
            sub["median_delta_fde"],
            marker="o",
            ms=2.5,
            lw=1.0,
            color=FEATURE_COLORS.get(feature),
            label=feature,
        )
    axes[0].set_xticks(np.arange(len(x_labels)))
    axes[0].set_xticklabels(x_labels, rotation=60, ha="right")
    axes[0].set_ylabel("Median delta FDE")
    axes[0].set_title("Chronological session trend")
    axes[0].legend(frameon=False, loc="best")

    region_summary = (
        summary_source.groupby(["brain_region", "feature_group"], as_index=False)
        .agg(frac_significant=("is_significant", "mean"), median_delta_fde=("delta_fde", "median"), n_units=("unit", "nunique"))
        .query("brain_region != 'Unknown'")
    )
    region_summary["sort_region"] = region_summary["brain_region"].str.contains("CA|DG|HP|HPC|mPFC|PFC|PrL|IL", case=False, regex=True)
    region_plot = (
        region_summary.sort_values(["sort_region", "median_delta_fde"], ascending=[False, False])
        .head(12)
        .copy()
    )
    if region_plot.empty:
        axes[1].text(0.5, 0.5, "No region labels", ha="center", va="center")
        axes[1].axis("off")
    else:
        labels = region_plot["brain_region"] + " | " + region_plot["feature_group"]
        axes[1].barh(np.arange(len(region_plot)), region_plot["median_delta_fde"], color="#6D5BA6")
        axes[1].set_yticks(np.arange(len(region_plot)))
        axes[1].set_yticklabels(labels)
        axes[1].invert_yaxis()
        axes[1].set_xlabel("Median delta FDE")
        axes[1].set_title("Region x feature summary")

save_figure(fig, "fig07_session_region_summary")
plt.show()
if not summary_source.empty:
    display(session_feature_summary.head(20))


## Concise Findings


In [ ]:
summary_lines = []
if not interval_summary.empty:
    top_interval = interval_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Primary interval GLMs**: `{top_interval.feature_group}` contributed most in `{top_interval.interval_label}` "
        f"(median delta FDE {top_interval.median_delta_fde:.4f}; standalone FDE {top_interval.median_standalone_fde:.4f}; "
        f"{100 * top_interval.frac_significant:.1f}% significant)."
    )
if not choice_time.empty:
    top_choice = choice_time.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Stop/pass timing**: strongest stop/pass contribution near `{top_choice.interval_label}` at {top_choice.rel_time_s:.2f} s "
        f"(median delta FDE {top_choice.median_delta_fde:.4f}; {100 * top_choice.frac_significant:.1f}% significant)."
    )
if not movement_decomp_summary.empty:
    top_move = movement_decomp_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Movement decomposition**: `{MOVEMENT_LABELS.get(top_move.feature_group, top_move.feature_group)}` was the strongest movement sub-block in `{top_move.interval_label}` "
        f"(median delta FDE {top_move.median_delta_fde:.4f})."
    )
if not movement_variable_summary.empty:
    top_var = movement_variable_summary.sort_values(["median_delta_fde", "frac_significant"], ascending=[False, False]).iloc[0]
    summary_lines.append(
        f"- **Movement variable**: `{MOVEMENT_LABELS.get(top_var.feature_group, top_var.feature_group)}` was the strongest single movement variable in `{top_var.interval_label}` "
        f"(median delta FDE {top_var.median_delta_fde:.4f})."
    )
if not all_feature_results.empty:
    n_sig = int(all_feature_results["is_significant"].sum())
    n_all = int(len(all_feature_results))
    summary_lines.append(f"- **All screens**: {n_sig:,}/{n_all:,} tests passed q < {FDR_ALPHA} with positive held-out delta FDE.")
if QUICK_TEST:
    summary_lines.append("- **Mode**: this was a capped quick validation run. Set `QUICK_TEST = False` and re-run for the full analysis.")
if not summary_lines:
    summary_lines.append("- No eligible GLM results were produced with the current settings.")

display(Markdown("\n".join(summary_lines)))


## Validation Checks


In [ ]:
required_t0_cols = {
    "session_id",
    "trial_id",
    "t0_event_name",
    "t0",
    "choice_R1",
    "choice_R2",
    *INTERVAL_SPECS.keys(),
    *(f"{name}_bins" for name in INTERVAL_SPECS),
}
required_behavior_cols = {
    "session_id",
    "from_ephys_timestamp",
    "trial_id",
    "frame_raw_500msMedian",
    "frame_YawPitch_abs_vel_sum_500msMedian",
    "frame_raw_abs_acc_500msMedian",
    "frame_YawPitch_abs_acc_sum_500msMedian",
    "forward_vs_rotation_corr",
    "frame_forward_prop",
    "head_angle",
    "head_angle_vel",
    "movement_energy_smooth5",
    "cue_visible",
    "track_zone_int",
    "frame_position",
}
required_fr_cols = {"session_id", "from_ephys_timestamp", *unit_cols}
missing_t0 = required_t0_cols - set(t0_events.columns)
missing_behavior = required_behavior_cols - set(behavior_40.columns)
missing_fr = required_fr_cols - set(fr40.columns)
assert not missing_t0, f"TrialWiseT0Events40ms missing columns: {sorted(missing_t0)}"
assert not missing_behavior, f"Behavior40msAligned missing columns: {sorted(missing_behavior)}"
assert not missing_fr, f"FiringRate40msHz missing columns: {sorted(missing_fr)}"
assert len(unit_cols) > 0, "No unit firing-rate columns found."
assert not event_bin_table.empty, "Event-aligned model table is empty."
assert set(event_bin_table["interval_name"].unique()).issubset(set(INTERVAL_ORDER)), "Unexpected intervals are present."
assert set(INTERVAL_ORDER).issubset(set(event_bin_table["interval_name"])), "At least one requested interval has no event-aligned rows."
assert set(coverage["interval_name"].astype(str)).issubset(set(INTERVAL_ORDER)), "Coverage includes intervals outside the requested three."
assert np.allclose(event_bin_table["exposure"].dropna().unique(), BIN_SECONDS), "Exposure must be 40 ms for all rows."

r1_rows = event_bin_table["interval_name"].eq("R1_entry_interval")
r2_rows = event_bin_table["interval_name"].eq("R2_entry_interval")
cue_rows = event_bin_table["interval_name"].eq("cue_entry_interval")
assert np.allclose(event_bin_table.loc[r1_rows, "choice_stop"], event_bin_table.loc[r1_rows, "choice_R1"], equal_nan=True), "R1 choice_stop must equal choice_R1."
assert np.allclose(event_bin_table.loc[r2_rows, "choice_stop"], event_bin_table.loc[r2_rows, "choice_R2"], equal_nan=True), "R2 choice_stop must equal choice_R2."
assert event_bin_table.loc[cue_rows, "choice_stop"].isna().all(), "Cue-entry choice_stop should be missing/excluded."
assert not interval_results.query("interval_name == 'cue_entry_interval' and feature_group == 'choice_stop'").shape[0], "Cue-entry stop/pass fits should not exist."

for unit in unit_cols:
    values = event_bin_table[unit]
    assert values.notna().all(), f"{unit} contains NaN spike counts."
    assert values.ge(0).all(), f"{unit} contains negative spike counts."
    assert np.allclose(values, np.rint(values)), f"{unit} contains non-integer spike counts."
assert set(all_feature_results["family"].dropna().unique()).issubset({"negbin"})
assert not all_feature_results.empty, "No GLM result rows were produced."
if not all_feature_results.empty:
    assert all_feature_results["q_value"].dropna().between(0, 1).all(), "q-values outside [0, 1]."
    for col in ["full_fde_cv", "reduced_fde_cv", "standalone_fde_cv", "delta_fde"]:
        assert all_feature_results[col].dropna().map(np.isfinite).all(), f"Non-finite {col} values."
        assert all_feature_results[col].dropna().abs().le(MAX_ABS_FDE).all(), f"Implausibly large {col} values."
print("Validation passed.")
